In [12]:
from modules.RailwaySimulationGenerator import RailwaySimulationGenerator
# Importing the global libraries
import importlib, sys, os, pandas as pd
# from dotenv import load_dotenv
import pyspark.sql.types as T
import pyspark.sql as sql
import pyspark.sql.functions as F
import numpy as np
import datetime as dt

os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.security.manager=allow"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.extraJavaOptions=-Djava.security.manager=allow pyspark-shell"

#Mandatory
importlib.reload(importlib)
importlib.reload(sys.modules["modules.RailwaySimulationGenerator"])
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
START_DATE : str = "2025-01-01"						# Simulation start date
START_TIME : str = "06:00:00"						# Simulation start time
NB_DAYS : int = 5									# Number of days to simulate
EDGES_MAX_SPEED : float = 27.78						# Max speed of the edges in m/s (120 km/h)
TRAIN_SPEED : float = 33.33							# Train speed in m/s (120 km/h)
DIRECTORY : str = "sumo_data"
OUTPUT_DIRECTORY : str = "new_start"

filenames = {
	"stations" : f"{DIRECTORY}/stations.csv",
	"edges" : f"{DIRECTORY}/station_to_station.csv",
	"punctuality" : f"{DIRECTORY}/punctuality/202501.csv",
    "trains" : f"{DIRECTORY}/trains.add.xml"
}

# sim_stations = [
# 	288,	# Cour-sur-Heure
# 	147, 	# Berzée
# ]

# sim_stations = [
# 	291,	# Couvin
# 	798, 	# Mariembourg
# 	961, 	# Philippeville
# ]

sim_stations = [
    291,
    798,
    961,
    1254,
    1207,
    976,
    147,
    288,
    133,
    612,
	976
]

sparkSession : sql.SparkSession = (sql.SparkSession.builder
	.appName("RailwaySimulationGenerator")
	.config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
	.getOrCreate()
)

rsg : RailwaySimulationGenerator = RailwaySimulationGenerator(
	stations_file = filenames["stations"],
    station_to_station_file = filenames["edges"],
    punctuality_data_file = filenames["punctuality"],
    output_dir = OUTPUT_DIRECTORY,
    train_speed = TRAIN_SPEED,
    nb_days = NB_DAYS,
    edge_max_speed = EDGES_MAX_SPEED,
    start_datetime = f"{START_DATE} {START_TIME}",
    spark = sparkSession, 
    sim_stations = sim_stations 
)

In [21]:
rsg.loadData()

Loading data...
Loading stations data...
Loading station to station data...
Loading punctuality data...
Data loaded successfully


In [22]:
rsg.filterData()

Filtering data...
Filtering stations...
Filtering edges...
Filtering punctuality data...
Data filtered successfully


In [153]:
# rsg.stations_df.printSchema()
# rsg.stations_df.show(5)

In [154]:
# rsg.station_to_station_df.printSchema()
# rsg.station_to_station_df.show(5)

In [23]:
rsg.punctuality_data_df.printSchema()
print(rsg.punctuality_data_df.count())
rsg.punctuality_data_df.show(100)

root
 |-- TRAIN_NO: integer (nullable = true)
 |-- RELATION: string (nullable = true)
 |-- TRAIN_SERV: string (nullable = true)
 |-- STOPPING_PLACE_ID: integer (nullable = true)
 |-- LINE_NO_DEP: string (nullable = true)
 |-- DELAY_ARR: integer (nullable = true)
 |-- DELAY_DEP: integer (nullable = true)
 |-- RELATION_DIRECTION: string (nullable = true)
 |-- LINE_NO_ARR: string (nullable = true)
 |-- REAL_DATE_ARR: string (nullable = true)
 |-- REAL_DATE_DEP: string (nullable = true)
 |-- PLANNED_DATETIME_ARR: timestamp (nullable = true)
 |-- PLANNED_DATETIME_DEP: timestamp (nullable = true)
 |-- NEXT_STOPPING_PLACE_ID: integer (nullable = true)

1544
+--------+--------+----------+-----------------+-----------+---------+---------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|  RELATION_DIRECTION|LINE_NO_ARR|REAL_DATE_ARR|REAL_DA

In [157]:
rsg.writeNetworkFiles()

In [158]:
rsg.generateNetwork(launch=False)

netconvert --node-files new_start/stations.nod.xml --edge-files new_start/edges.edg.xml --railway.signal.guess.by-stops true --output-file new_start/network.net.xml --proj.utm true


In [159]:
rsg.generateTrips()

Processing trip number 3927
Processing info {'departure_station': 291, 'arrival_station': 798, 'sumo_time': 2400} with mapping False and False
Adding info {'departure_station': 291, 'arrival_station': 798, 'sumo_time': 2400}
Processing info {'departure_station': 798, 'arrival_station': 2199, 'sumo_time': 2940} with mapping False and True
Adding info {'departure_station': 798, 'arrival_station': 961, 'sumo_time': 2940}
Processing info {'departure_station': 2199, 'arrival_station': 961, 'sumo_time': 3480} with mapping True and False
Adding info {'departure_station': 798, 'arrival_station': 961, 'sumo_time': 3480}
Processing info {'departure_station': 961, 'arrival_station': 1254, 'sumo_time': 3720} with mapping False and False
Adding info {'departure_station': 961, 'arrival_station': 1254, 'sumo_time': 3720}
Processing info {'departure_station': 1254, 'arrival_station': 1207, 'sumo_time': 4200} with mapping False and False
Adding info {'departure_station': 1254, 'arrival_station': 1207, 

In [160]:
for trip in rsg.trips :
	print(trip)

[{'departure_station': 291, 'arrival_station': 798, 'sumo_time': 2400}, {'departure_station': 798, 'arrival_station': 961, 'sumo_time': 3210.0}, {'departure_station': 961, 'arrival_station': 1254, 'sumo_time': 3720}, {'departure_station': 1254, 'arrival_station': 1207, 'sumo_time': 4200}, {'departure_station': 1207, 'arrival_station': 976, 'sumo_time': 4620}, {'departure_station': 976, 'arrival_station': 147, 'sumo_time': 4860}, {'departure_station': 147, 'arrival_station': 288, 'sumo_time': 5040}, {'departure_station': 288, 'arrival_station': 514, 'sumo_time': 5280}, {'departure_station': 514, 'arrival_station': 133, 'sumo_time': 5460}, {'departure_station': 133, 'arrival_station': 612, 'sumo_time': 5580}]
[{'departure_station': 259, 'arrival_station': 612, 'sumo_time': 8100}, {'departure_station': 612, 'arrival_station': 133, 'sumo_time': 8700}, {'departure_station': 133, 'arrival_station': 514, 'sumo_time': 8880}, {'departure_station': 514, 'arrival_station': 288, 'sumo_time': 9060}

In [161]:
rsg.writeScheduleFiles()

In [162]:
rsg.writeConfigurationFile()

In [163]:
rsg.startSimulation(launch=False)

sumo -c new_start/config.sumocfg
